In [2]:
'''
    构建关键词表
    原始日志  ->  去除动态变量  ->  剩余固定语义 token  ->  构建词典
    格式为
    {"word_set": [关键词1, 关键词2, ...]}
'''

'\n    构建关键词表\n    格式为\n    {"word_set": [关键词1, 关键词2, ...]}\n'

In [1]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
import os
import pandas as pd
import json
import pickle
import re

In [2]:
def extract_istio_words(
    data_base: str,
    all_entity_list: list,
    rename_pod2service,
    out_istio_words_json: str,
    stopwords_regex_list: list,
    skip_if_path_contains=("test_data",),
    chunksize: int = 100_000,
):
    """
    Scan all `log_filebeat-testbed-log-envoy.csv` under `data_base` (excluding paths containing `skip_if_path_contains`),
    tokenize log messages after regex-based stopword removal, and export a global word vocabulary.

    Output JSON format:
        {"word_set": [word1, word2, ...]}
    """

    target_name = "log_filebeat-testbed-log-envoy.csv"

    # 用 set 去重，速度快；最后再转 list 写 json
    word_set = set()

    for root, _, files in os.walk(data_base):
        if target_name not in files:
            continue

        log_path = os.path.join(root, target_name)

        # 过滤 test_data
        if any(x in log_path for x in skip_if_path_contains):
            continue

        reader = pd.read_csv(log_path, chunksize=chunksize)

        for chunk in reader:
            # 用 itertuples 更快，且可以 getattr
            for row in chunk.itertuples(index=False):
                cmdb_id = getattr(row, "cmdb_id", None)
                value = getattr(row, "value", None)

                if cmdb_id is None or value is None:
                    continue
                if cmdb_id not in all_entity_list:
                    continue
                if not isinstance(value, str):
                    continue

                # 与原逻辑一致：先清洗，再按空格切词
                new_str = value.strip()
                for stop_word in stopwords_regex_list:
                    new_str = re.sub(stop_word, "", new_str)

                # 这里不需要计数，只要词表，所以直接 set.update
                for w in new_str.split(" "):
                    if w:
                        word_set.add(w)

    # 写出
    os.makedirs(os.path.dirname(out_istio_words_json), exist_ok=True)
    with open(out_istio_words_json, "w", encoding="utf-8") as f:
        json.dump({"word_set": sorted(word_set)}, f, ensure_ascii=False, indent=2)

    print(f"[OK] Already extract istio_words -> {out_istio_words_json}")
    return {"word_set": sorted(word_set)}

In [3]:
stopwords_regex_list = [
        " \"*([0-9]{1,3}\\.[0-9]{1,3}\\.[0-9]{1,3}\\.[0-9]{1,3})\\:?([0-9]{1,5})?\"*",
        "\"(\\w|\\d)*-(\\w|\\d)*-(\\w|\\d)*-(\\w|\\d)*-(\\w|\\d)*\"",
        "( |^)\"*(GET|POST|PUT|DELETE)\"*",
        " \"*[0-9]+\"*",
        " (\"*grpc-(\\w)*/)[0-9]+.[0-9]+.[0-9]+\"*",
        " (\"*Go(-(\\w)*)*/)[0-9]+(.[0-9]+)*\"*",
        "( |^)\"*-\"*",
        " (default|(\\(manylinux; chttp2\\))|(HTTP/\\d(.\\d)*\"*))\"*"
    ]

In [4]:
all_entity_list = [
    'node-1', 
    'node-2', 
    'node-3', 
    'node-4', 
    'node-5', 
    'node-6', 
    'adservice', 
    'cartservice', 
    'checkoutservice', 
    'currencyservice', 
    'emailservice', 
    'frontend', 
    'paymentservice', 
    'productcatalogservice', 
    'recommendationservice', 
    'shippingservice', 
    'adservice-0', 
    'adservice-1', 
    'adservice-2', 
    'adservice2-0', 
    'cartservice-0', 
    'cartservice-1', 
    'cartservice-2', 
    'cartservice2-0', 
    'checkoutservice-0', 
    'checkoutservice-1', 
    'checkoutservice-2', 
    'checkoutservice2-0', 
    'currencyservice-0', 
    'currencyservice-1', 
    'currencyservice-2', 
    'currencyservice2-0', 
    'emailservice-0', 
    'emailservice-1', 
    'emailservice-2', 
    'emailservice2-0', 
    'frontend-0', 
    'frontend-1', 
    'frontend-2', 
    'frontend2-0', 
    'paymentservice-0', 
    'paymentservice-1', 
    'paymentservice-2', 
    'paymentservice2-0', 
    'productcatalogservice-0', 
    'productcatalogservice-1', 
    'productcatalogservice-2', 
    'productcatalogservice2-0', 
    'recommendationservice-0', 
    'recommendationservice-1', 
    'recommendationservice-2', 
    'recommendationservice2-0', 
    'shippingservice-0', 
    'shippingservice-1', 
    'shippingservice-2', 
    'shippingservice2-0'
]

In [5]:
data_base = "/home/ZZData/Eadro/preprocess/datasets/aiops2022/"

def rename_pod2service(pod_name):
    return pod_name.replace('2-0', '').replace('-0', '').replace('-1', '').replace('-2', '')

extract_istio_words(
    data_base=data_base,
    all_entity_list=all_entity_list,
    rename_pod2service=rename_pod2service,   # 这里其实没用到也没关系，你想统一接口可以保留
    stopwords_regex_list=stopwords_regex_list,
    out_istio_words_json="../../temp_data/aiops22/analysis/log/istio_words.json",
)

[OK] Already extract istio_words -> ../../temp_data/aiops22/analysis/log/istio_words.json


{'word_set': ['"Prometheus/2.12.0"',
  '"Python/THttpClient',
  '"adservice.ts:8088"',
  '"adservice:9555"',
  '"cartservice:7070"',
  '"checkoutservice:5050"',
  '"currencyservice:7000"',
  '"emailservice:5000"',
  '"frontend.ts:80"',
  '"jaeger-collector:14268"',
  '"jaeger-collector:9411"',
  '"k6/0.26.2',
  '"okhttp/3.14.7"',
  '"paymentservice:50051"',
  '"productcatalogservice:3550"',
  '"recommendationservice:8080"',
  '"shippingservice:50051"',
  '"transport:',
  '(email_server.py)"',
  '(https://k6.io/)"',
  '(recommendation_server.py)"',
  '/',
  '/api/traces',
  '/api/traces?format=jaeger.thrift',
  '/api/v2/spans',
  '/cart',
  '/cart/checkout',
  '/hipstershop.AdService/GetAds',
  '/hipstershop.CartService/AddItem',
  '/hipstershop.CartService/EmptyCart',
  '/hipstershop.CartService/GetCart',
  '/hipstershop.CheckoutService/PlaceOrder',
  '/hipstershop.CurrencyService/Convert',
  '/hipstershop.CurrencyService/GetSupportedCurrencies',
  '/hipstershop.EmailService/SendOrderC

**根因定位**

In [6]:
import os
import math
import copy
import pickle
import numpy as np
from typing import Any, Dict, List, Tuple, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import Parameter, ParameterDict
from torch_geometric.nn import MessagePassing

from shared_util.seed import seed_everything
from shared_util.logger import Logger
from shared_util.file_handler import FileHandler

from HolisticRCA.model.common.GAT_net import GATNet
from HolisticRCA.util.ent_batch_graph import EntBatchGraph


# =========================================================
# 0) small utils
# =========================================================
def load_pkl(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_pkl(obj: Any, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)


def copy_batch_data(batch_data: Dict[str, Any], device: Optional[torch.device] = None):
    out = {}
    for k, v in batch_data.items():
        if torch.is_tensor(v):
            out[k] = v.clone()
            if device is not None:
                out[k] = out[k].to(device)
        else:
            out[k] = copy.deepcopy(v)
    return out


def safe_sigmoid_np(x: torch.Tensor) -> np.ndarray:
    return torch.sigmoid(x).detach().cpu().numpy()


ModuleNotFoundError: No module named 'shared_util'

In [ ]:
class PositionalEmbedding(nn.Module):
    def __init__(self, in_features, num_of_o11y_features, register_name):
        super().__init__()

        temp_in_features = in_features
        if temp_in_features % 2 == 1:
            temp_in_features += 1

        temp_num_of_o11y_features = num_of_o11y_features
        if temp_num_of_o11y_features % 2 == 1:
            temp_num_of_o11y_features += 1

        pe = torch.zeros(temp_in_features, temp_num_of_o11y_features).float()
        pe.require_grad = False

        position = torch.arange(0, temp_in_features).float().unsqueeze(1)
        div_term = (
            torch.arange(0, temp_num_of_o11y_features, 2).float()
            * -(math.log(10000.0) / temp_num_of_o11y_features)
        ).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.transpose(1, 0)[:num_of_o11y_features, :in_features]
        self.register_buffer("pe", pe)

    def forward(self, x):
        return self.pe + x


class RepresentationLearning(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data
        self.different_modal_mapping_dict = nn.ModuleDict()
        self.positional_embedding_dict = nn.ModuleDict()
        self.modal_transformer_encoder_layer_dict = nn.ModuleDict()

        for modal_type in self.meta_data["modal_types"]:
            self.positional_embedding_dict[modal_type] = PositionalEmbedding(
                in_features=param_dict["window_size"],
                num_of_o11y_features=self.meta_data["o11y_length"][modal_type],
                register_name=f"{modal_type}_pe",
            )
            self.different_modal_mapping_dict[modal_type] = nn.Linear(
                in_features=param_dict["window_size"],
                out_features=param_dict["orl_te_in_channels"],
            )
            transformer_encoder_layer = nn.TransformerEncoderLayer(
                d_model=param_dict["orl_te_in_channels"],
                nhead=param_dict["orl_te_heads"],
            )
            self.modal_transformer_encoder_layer_dict[modal_type] = nn.TransformerEncoder(
                transformer_encoder_layer,
                num_layers=param_dict["orl_te_layers"],
            )

    def forward(self, batch_data):
        for modal_type in self.meta_data["modal_types"]:
            batch_data[f"x_{modal_type}"] = batch_data[f"x_{modal_type}"].to(self.device_marker.device)
            batch_data[f"x_{modal_type}"] = self.positional_embedding_dict[modal_type](
                batch_data[f"x_{modal_type}"]
            ).contiguous()
            batch_data[f"x_{modal_type}"] = self.different_modal_mapping_dict[modal_type](
                batch_data[f"x_{modal_type}"]
            )
            batch_data[f"x_{modal_type}"] = self.modal_transformer_encoder_layer_dict[modal_type](
                batch_data[f"x_{modal_type}"]
            )
        return batch_data


class FeatureIntegration(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data

        self.ent_transformer_encoder_layer_dict = nn.ModuleDict()
        self.ent_feature_align_dict = nn.ModuleDict()

        in_dim = param_dict["efi_in_dim"]

        for ent_type in self.meta_data["ent_types"]:
            all_ent_feature_length = 0
            for modal_type in self.meta_data["modal_types"]:
                all_ent_feature_length += self.meta_data["max_ent_feature_num"][ent_type][modal_type]

            transformer_encoder_layer = nn.TransformerEncoderLayer(
                d_model=in_dim,
                nhead=param_dict["efi_te_heads"],
            )
            self.ent_transformer_encoder_layer_dict[ent_type] = nn.TransformerEncoder(
                transformer_encoder_layer,
                num_layers=param_dict["efi_te_layers"],
            )

            self.ent_feature_align_dict[ent_type] = nn.Linear(
                all_ent_feature_length * in_dim,
                param_dict["efi_out_dim"],
            )

    def forward(self, batch_data):
        batch_size = batch_data["y"].shape[0]

        x_ent = []
        for ent_type in self.meta_data["ent_types"]:
            for ent_index in range(
                self.meta_data["ent_type_index"][ent_type][0],
                self.meta_data["ent_type_index"][ent_type][1],
            ):
                x = []
                for modal_type in self.meta_data["modal_types"]:
                    feature_index_pair = self.meta_data["ent_features"][modal_type][ent_index][1]
                    modal_data = batch_data[f"x_{modal_type}"][
                        :, feature_index_pair[0]:feature_index_pair[1], :
                    ]
                    padding = torch.zeros(
                        batch_size,
                        self.meta_data["max_ent_feature_num"][ent_type][modal_type] - modal_data.shape[1],
                        modal_data.shape[2],
                    ).to(self.device_marker.device)
                    modal_data = torch.cat((modal_data, padding), 1)
                    x.append(modal_data)

                x = torch.cat(x, dim=1)
                x = self.ent_transformer_encoder_layer_dict[ent_type](x)
                x = x.view(batch_size, x.shape[1] * x.shape[2]).contiguous()
                x = self.ent_feature_align_dict[ent_type](x)
                x_ent.append(x)

        x_ent = torch.stack(x_ent, dim=1)
        batch_data["x_ent"] = x_ent
        return batch_data


class FeatureFusion(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data
        self.GAT_net = GATNet(
            in_channels=param_dict["eff_in_dim"],
            out_channels=param_dict["eff_GAT_out_channels"],
            heads=param_dict["eff_GAT_heads"],
            dropout=param_dict["eff_GAT_dropout"],
        )

    def forward(self, batch_data):
        ent_batch_graph = EntBatchGraph(batch_data, self.meta_data).to(self.device_marker.device)
        x = ent_batch_graph.x["re"]
        x = self.GAT_net(x, ent_batch_graph.edge_index)
        return x


class FaultClassifier(nn.Module):
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.device_marker = nn.Parameter(torch.empty(0))
        self.meta_data = meta_data
        self.linear_dict = nn.ModuleDict()
        for ent_type in self.meta_data["ent_types"]:
            index_pair = self.meta_data["ent_fault_type_index"][ent_type]
            self.linear_dict[ent_type] = nn.Linear(
                param_dict["eff_GAT_out_channels"],
                index_pair[1] - index_pair[0],
            )

    def forward(self, x):
        output = dict()
        for ent_type in self.meta_data["ent_types"]:
            temp = x[
                :,
                self.meta_data["ent_type_index"][ent_type][0]:self.meta_data["ent_type_index"][ent_type][1],
                :
            ]
            temp = temp.reshape(temp.shape[0] * temp.shape[1], temp.shape[2]).contiguous()
            output[ent_type] = self.linear_dict[ent_type](temp)
        return output

class HolisticRCAModel(nn.Module):
    """
    不用 nn.Sequential，显式命名，方便 explainer 访问 self.feature_fusion.GAT_net
    """
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.representation_learning = RepresentationLearning(param_dict, meta_data)
        self.feature_integration = FeatureIntegration(param_dict, meta_data)
        self.feature_fusion = FeatureFusion(param_dict, meta_data)
        self.fault_classifier = FaultClassifier(param_dict, meta_data)

    def forward(self, batch_data):
        batch_data = self.representation_learning(batch_data)
        batch_data = self.feature_integration(batch_data)
        x = self.feature_fusion(batch_data)
        out = self.fault_classifier(x)
        return out


In [ ]:
class HolisticRCAModel(nn.Module):
    """
    不用 nn.Sequential，显式命名，方便 explainer 访问 self.feature_fusion.GAT_net
    """
    def __init__(self, param_dict, meta_data):
        super().__init__()
        self.representation_learning = RepresentationLearning(param_dict, meta_data)
        self.feature_integration = FeatureIntegration(param_dict, meta_data)
        self.feature_fusion = FeatureFusion(param_dict, meta_data)
        self.fault_classifier = FaultClassifier(param_dict, meta_data)

    def forward(self, batch_data):
        batch_data = self.representation_learning(batch_data)
        batch_data = self.feature_integration(batch_data)
        x = self.feature_fusion(batch_data)
        out = self.fault_classifier(x)
        return out


In [ ]:
def rearrange_y(meta_data, y, device):
    """
    return:
      y_dict[ent_type] = (B, n_ent, K_local)
    """
    if y.dim() == 3 and y.shape[-1] == 1:
        y = y.squeeze(-1)
    if y.dim() != 2:
        raise ValueError(f"Expect y shape (B,E), got {tuple(y.shape)}")

    y = y.to(device).long()
    y_dict = {}

    for ent_type in meta_data["ent_types"]:
        es, ee = meta_data["ent_type_index"][ent_type]
        fs, fe = meta_data["ent_fault_type_index"][ent_type]
        n_ent = ee - es
        k_local = fe - fs

        temp = torch.zeros((y.shape[0], n_ent, k_local), dtype=torch.float32, device=device)
        temp_y = y[:, es:ee]  # (B, n_ent)

        # 0 表示正常，>0 表示全局 fault id，从 1 开始
        abnormal_pos = torch.nonzero(temp_y > 0, as_tuple=False)
        for pos in abnormal_pos:
            b, e_local = int(pos[0]), int(pos[1])
            global_fault_id = int(temp_y[b, e_local].item()) - 1  # 转成 0-based
            if fs <= global_fault_id < fe:
                local_fault_id = global_fault_id - fs
                temp[b, e_local, local_fault_id] = 1.0

        y_dict[ent_type] = temp

    return y_dict

In [ ]:
def rearrange_y(meta_data, y, device):
    """
    return:
      y_dict[ent_type] = (B, n_ent, K_local)
    """
    if y.dim() == 3 and y.shape[-1] == 1:
        y = y.squeeze(-1)
    if y.dim() != 2:
        raise ValueError(f"Expect y shape (B,E), got {tuple(y.shape)}")

    y = y.to(device).long()
    y_dict = {}

    for ent_type in meta_data["ent_types"]:
        es, ee = meta_data["ent_type_index"][ent_type]
        fs, fe = meta_data["ent_fault_type_index"][ent_type]
        n_ent = ee - es
        k_local = fe - fs

        temp = torch.zeros((y.shape[0], n_ent, k_local), dtype=torch.float32, device=device)
        temp_y = y[:, es:ee]  # (B, n_ent)

        # 0 表示正常，>0 表示全局 fault id，从 1 开始
        abnormal_pos = torch.nonzero(temp_y > 0, as_tuple=False)
        for pos in abnormal_pos:
            b, e_local = int(pos[0]), int(pos[1])
            global_fault_id = int(temp_y[b, e_local].item()) - 1  # 转成 0-based
            if fs <= global_fault_id < fe:
                local_fault_id = global_fault_id - fs
                temp[b, e_local, local_fault_id] = 1.0

        y_dict[ent_type] = temp

    return y_dict

In [ ]:
class Explainer(nn.Module):
    def __init__(self, model, meta_data, param_dict):
        super().__init__()
        self.coeffs = {
            "ent_edge_size": 0.005,
            "ent_edge_reduction": "sum",
            "o11y_size": 1.0,
            "o11y_reduction": "mean",
            "ent_edge_entropy": 1.0,
            "o11y_entropy": 0.1,
            "EPS": 1e-15,
        }
        self.device = torch.device(
            "cuda" if torch.cuda.is_available() and param_dict["explainer_gpu"] else "cpu"
        )
        self.model = model
        self.meta_data = meta_data
        self.param_dict = param_dict
        self.o11y_mask = self.hard_o11y_mask = None
        self.ent_edge_mask = self.hard_ent_edge_mask = None
        self.criterion = torch.nn.BCEWithLogitsLoss()
        self.logger = Logger(logging_level="DEBUG").logger

    def init_o11y_mask(self):
        mask = ParameterDict()
        for modal_type in self.meta_data["modal_types"]:
            mask[modal_type] = Parameter(
                torch.FloatTensor(self.meta_data["o11y_length"][modal_type]).to(self.device)
            )
            std = 0.1
            with torch.no_grad():
                mask[modal_type].normal_(1.0, std)
        self.o11y_mask = mask

    def init_ent_edge_mask(self, test_sample_data):
        num_edges = test_sample_data["ent_edge_index"].shape[2]
        num_entities = len(self.meta_data["ent_names"])
        std = nn.init.calculate_gain("relu") * math.sqrt(2.0 / (num_entities + num_entities))
        mask = Parameter(torch.randn(num_edges, device=self.device) * std)
        self.ent_edge_mask = mask

    def set_masks(self, test_sample_data):
        edge_index = torch.squeeze(test_sample_data["ent_edge_index"], dim=0)
        for module in self.model.feature_fusion.GAT_net.modules():
            loop_mask = torch.full_like(edge_index[0], True, dtype=torch.bool)
            if isinstance(module, MessagePassing):
                module.explain = True
                module._edge_mask = self.ent_edge_mask
                module._loop_mask = loop_mask
                module._apply_sigmoid = True

    def clean_explainer(self):
        for module in self.model.feature_fusion.GAT_net.modules():
            if isinstance(module, MessagePassing):
                module.explain = False
                module._edge_mask = None
                module._loop_mask = None
                module._apply_sigmoid = True
        self.o11y_mask = self.hard_o11y_mask = None
        self.ent_edge_mask = self.hard_ent_edge_mask = None

    def apply_o11y_mask(self, batch_data):
        """
        batch_data[x_modal]: (B, F_modal, T)
        用 feature-wise mask 去乘第 2 维 F
        """
        for modal_type in self.meta_data["modal_types"]:
            m = self.o11y_mask[modal_type].sigmoid().view(1, -1, 1)
            batch_data[f"x_{modal_type}"] = batch_data[f"x_{modal_type}"] * m
        return batch_data

    def _loss(self, masked_pred, original_y_pred):
        """
        保持原来的 fault type prediction
        """
        loss = self.criterion(masked_pred, original_y_pred.detach())

        # edge sparsity
        ent_edge_mask = self.ent_edge_mask.sigmoid()
        if self.coeffs["ent_edge_reduction"] == "sum":
            edge_reduce = ent_edge_mask.sum()
        else:
            edge_reduce = ent_edge_mask.mean()
        loss = loss + self.coeffs["ent_edge_size"] * edge_reduce

        # edge entropy
        ent_edge_entropy = -ent_edge_mask * torch.log(ent_edge_mask + self.coeffs["EPS"]) - \
                           (1 - ent_edge_mask) * torch.log(1 - ent_edge_mask + self.coeffs["EPS"])
        loss = loss + self.coeffs["ent_edge_entropy"] * ent_edge_entropy.mean()

        # o11y sparsity / entropy
        o11y_reduce_list = []
        o11y_entropy_list = []
        for modal_type in self.meta_data["modal_types"]:
            m = self.o11y_mask[modal_type].sigmoid()
            if self.coeffs["o11y_reduction"] == "mean":
                o11y_reduce_list.append(m.mean())
            else:
                o11y_reduce_list.append(m.sum())

            ent = -m * torch.log(m + self.coeffs["EPS"]) - (1 - m) * torch.log(1 - m + self.coeffs["EPS"])
            o11y_entropy_list.append(ent.mean())

        loss = loss + self.coeffs["o11y_size"] * torch.stack(o11y_reduce_list).mean()
        loss = loss + self.coeffs["o11y_entropy"] * torch.stack(o11y_entropy_list).mean()

        return loss

    def train_explainer(self, test_sample_data, original_y_pred, entity_type, entity_index):
        """
        输入:
          original_y_pred: (K_local,) 这是 classifier 对某个 entity 的 logit
        输出:
          ent_name_result: [(score, ent_name), ...]
          o11y_name_result: [(score, o11y_name), ...]
        """
        self.init_o11y_mask()
        self.init_ent_edge_mask(test_sample_data)
        self.set_masks(test_sample_data)

        parameters = [self.ent_edge_mask]
        for key in self.o11y_mask.keys():
            parameters.append(self.o11y_mask[key])

        optimizer = torch.optim.Adam(
            parameters,
            lr=self.param_dict["explainer_lr"],
            weight_decay=self.param_dict["explainer_weight_decay"],
        )

        self.model.eval()

        for _ in range(self.param_dict["explainer_epochs"]):
            optimizer.zero_grad()

            masked_batch = copy_batch_data(test_sample_data, self.device)
            masked_batch = self.apply_o11y_mask(masked_batch)

            out = self.model(masked_batch)
            # out[entity_type] shape = (B*n_ent, K_local)
            pred = out[entity_type][entity_index]
            loss = self._loss(pred, original_y_pred)

            loss.backward()
            optimizer.step()

        # -------- produce explanation result --------
        o11y_name_result = []
        for modal_type in self.meta_data["modal_types"]:
            m = self.o11y_mask[modal_type].sigmoid().detach().cpu().numpy()
            names = self.meta_data["o11y_names"][modal_type]
            for i, score in enumerate(m):
                o11y_name_result.append((float(score), names[i]))

        o11y_name_result = sorted(o11y_name_result, key=lambda x: x[0], reverse=True)

        # entity importance: 用 edge mask 聚合成 entity score
        ent_scores = {name: 0.0 for name in self.meta_data["ent_names"]}
        edge_mask = self.ent_edge_mask.sigmoid().detach().cpu().numpy()
        edge_index = torch.squeeze(test_sample_data["ent_edge_index"], dim=0).detach().cpu().numpy()

        for ei in range(edge_index.shape[1]):
            src = int(edge_index[0, ei])
            dst = int(edge_index[1, ei])
            score = float(edge_mask[ei])
            ent_scores[self.meta_data["ent_names"][src]] += score
            ent_scores[self.meta_data["ent_names"][dst]] += score

        ent_name_result = sorted([(v, k) for k, v in ent_scores.items()], key=lambda x: x[0], reverse=True)

        self.clean_explainer()
        return ent_name_result, o11y_name_result


In [ ]:
if __name__ == "__main__":
    seed_everything()
    torch.use_deterministic_algorithms(True)

    data_base_path = "../../../../HolisticRCA"
    window_size = 10

    dataset_path = f"{data_base_path}/temp_data/aiops22/dataset/merge_multimodal/rca_multimodal_window_size_{window_size}.pkl"
    model_base_path = FileHandler.set_folder(f"{data_base_path}/model/aiops22")
    checkpoint_dir = FileHandler.set_folder(model_base_path + "/checkpoint")
    localization_dir = FileHandler.set_folder(model_base_path + "/localization")

    aiops22_params = {
        "dataset_path": dataset_path,
        "model_path": f"{checkpoint_dir}/main.pt",
        "output_path": f"{localization_dir}/localization_window_size_{window_size}.pkl",

        "window_size": window_size,

        "gpu": True,
        "batch_size": 1,

        "node_accuracy_th": 0.5,
        "service_accuracy_th": 0.5,
        "pod_accuracy_th": 0.5,

        "orl_te_heads": 2,
        "orl_te_layers": 2,
        "orl_te_in_channels": 256,

        "efi_in_dim": 256,
        "efi_te_heads": 4,
        "efi_te_layers": 2,
        "efi_out_dim": 256,

        "eff_in_dim": 256,
        "eff_GAT_out_channels": 128,
        "eff_GAT_heads": 2,
        "eff_GAT_dropout": 0.1,

        "ec_fault_types": 15,

        "explainer_gpu": True,
        "explainer_epochs": 100,
        "explainer_lr": 0.1,
        "explainer_weight_decay": 0.001,
    }

    localizer = Stage2Localizer(aiops22_params)
    localizer.predict()